_`r format(Sys.time(), '🗓️ %d %B, %Y 🕔 %H:%M')`_

In [ ]:
library(dplyr)
library(DT)
library(tidyr)
library(scales)

In [ ]:
infile <- params$infile
platform <- params$platform
data <- read.table(infile, header = T)
attention_df <- data %>% select(-1,-2) %>% rowwise() %>% summarise(count = sum(c(nameMismatch, noBX, badBX)))
attention <- sum(attention_df$count > 0)
noMItag <- sum(data$noMI == data$alignments)
noBXtag <- sum(data$noBX == data$alignments)
bxnotlast <- sum(data$bxNotLast > 0)

# File Checks
<h1> Preflight checks for BAM files </h1>

This report reflects the BAM files Harpy processed to identify obvious
[formatting issues](#details) that may require your attention.

## valueboxes

In [ ]:
list(
  icon = "files",
  color = "light",
  value = prettyNum(nrow(data), big.mark = ",")
)

In [ ]:
list(
  icon = ifelse(attention > 0, "exclamation-triangle", "check2-square"),
  color = ifelse(attention > 0, "warning", "success"),
  value = prettyNum(attention, big.mark = ",")
)

In [ ]:
list(
  icon = ifelse(noMItag > 0, "exclamation-triangle", "check2-square"),
  color = ifelse(noMItag > 0, "warning", "success"),
  value = prettyNum(noBXtag, big.mark = ",")
)

In [ ]:
list(
  icon = ifelse(noMItag > 0, "exclamation-triangle", "check2-square"),
  color = ifelse(noMItag > 0, "warning", "success"),
  value = prettyNum(noMItag, big.mark = ",")
)

In [ ]:
list(
  icon = ifelse(bxnotlast > 0, "exclamation-triangle", "check2-square"),
  color = ifelse(bxnotlast > 0, "warning", "success"),
  value = prettyNum(bxnotlast, big.mark = ",")
)

## viz desc

| column            | 🟦 pass condition 🟦                                                             | 🟨 fail condition 🟨                                       |
|:------------------|:---------------------------------------------------------------------------------|:-----------------------------------------------------------|
| **Name Mismatch** | the file name matches the `@RG ID:` tag in the header                            | file name does not match `@RG ID:` in the header           |
| **MI tag**        | **any** alignments with `BX:Z` tag also have `MI`                                | **all** reads have `BX:Z` tag present but `MI` tag absent  |
| **BX:Z tag**      | **any** `BX:Z` tags present                                                      | **all** alignments lack `BX:Z` tag                         |
| **BX:Z last**     | **all** reads have `BX:Z` as final tag in alignment records                      | **at least 1 read** doesn't have `BX:Z` tag as final tag   |
| **Format**        | **all** alignments with `BX:Z` tag have properly formatted `r platform` barcodes | **any** `BX:Z` barcodes have incorrect `r platform` format |

## file stats

In [ ]:
datatable(
  data,
  escape = F,
  rownames = F,
  filter = "top",
  extensions = 'Buttons',
  fillContainer = T,
  colnames = c("File", "Reads", "Name Mismatch", "MI tag", "BX:Z tag", "BX:Z last", "Format"),
  options = list(
    paging = F,
    scrollX = F,
    dom = 'Brtp',
    buttons = list(list(extend = "csv",filename = "bam_checks"))
  )
) %>%
formatStyle(
  columns = 3:ncol(data),
  # Blue for values <= 0, yellow for values > 0
  backgroundColor = styleInterval(0, c('#4faad1', '#feda75'))
) %>%
formatStyle(
  columns = 2,
  # yellow for values == 0, blue for values > 0
  backgroundColor = styleInterval(0, c('#feda75', '#4faad1'))
)

# Details
## interpret
::: {.card title="Interpreting the Data"}

The `harpy validate bam ...` command created a `filecheck.bam.tsv` file in the specified
output directory that summarizes the results that are included in this report.
This file contains a tab-delimited table with the columns: `file`,	`nameMismatch`, `alignments`,	`noBX`, and	`badBX`.
These columns are defined below and the severities of their issues are given as colored emoji:

- ⬜ is minor
- 🔶 is moderate
- 🛑 is critical

file
: The name of the BAM file

nameMismatch 🛑
: The sample name of the file inferred from name of the file (i.e. Harpy assumes `sample1.bam` to be `sample1`) does not match the `@RG ID` tag in the alignment file.

alignments
: The total number of alignment records in the file

noMI 🛑
: Alignment records lack `MI` tag (`MI:i` or `MI:Z`)

noBX ⬜
: The number of alignments that do not have a `BX:Z` tag in the record
If you expect all or some of your reads should have `BX:Z` tags, then further investigation is necessary

badBX 🛑
: The barcode in the `BX:Z` tag does not adhere to the proper `r platform` format

bxNotLast 🔶
: The `BX:Z` tag in the alignment record is not the last tag
Only relevant for LEVIATHAN variant calling and can be ignored if not intending to call
structural variants with LEVIATHAN,otherwise the `BX:Z` tag must be the last comment in an alignment
record. `harpy align` will automatically move the `BX:Z` tag to the end of the alignment record.
:::

### Proper format {.no-title}
::: {.card title="Proper BAM format"}

BX Tag
: A proper linked-read alignment file will contain a `BX:Z` tag with an alignment record
that features a properly-formatted barcode. The formats are:

- haplotagging: `AXXCXXBXXDXX` where `X` is a 0-9 integer
  - e.g. `A02C56B09D88`
  - invalidated: if any position is `00` (e.g. `A91C00B42D57`)
- stlfr: `X_X_X` where `X` is any integer
  - e.g. `9_851_22 
  - invalidated: if any position is `0` (e.g. `51_0_492`)
- tellseq/10x: `ATCGN` nucleotide letters
  - e.g. `ATATTTACGGGAC`
  - invalidated: if any nucleotide is `N` (e.g. `ATACANGGAT`)

If a barcode is not in that format, then it's likely the input FASTQ used for read mapping is the source of the
issue. You can check those FASTQ files for errors with `harpy validate fastq`. 

: Example Alignment Record
Below is an example of a proper alignment record for a file named `sample1.bam`.
Note the tag `RG:Z:sample1`, indicating this alignment is associated with `sample1` and
matches the file name. Also note the correctly formatted haplotagging barcode `BX:Z:A19C01B86D78`
and the presence of a `MI:` tag. To reduce horizontal scrolling, the alignment sequence and Phred
scores have been replaced with `SEQUENCE` and `QUALITY`, respectively.

```
A00814:267:HTMH3DRXX:2:1132:26268:10316 113     contig1 6312    60      4S47M1D86M      =       6312    0       SEQUENCE  QUALITY     NM:i:2  MD:Z:23C23^C86  MC:Z:4S47M1D86M AS:i:121        XS:i:86 RG:Z:sample1   MI:i:4040669       RX:Z:GAAACGATGTTGC+CCTAAGCCGAATC        QX:Z:FFFFFFFFFFFFF+FFFFF:FFFFFFF    BX:Z:A19C01B86D78
```

:::